# 01 — Data Understanding


> **Notebook 1 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11. Each one saves its results to disk so the next
> one can pick them up.

---

## 🎯 Goal of this notebook

Before writing a single line of machine learning, we must **understand what we are holding**.
This notebook answers four questions:

1. How much data do we have?
2. What does every column actually mean?
3. Is anything missing?
4. Is the data believable — or is it dirty?

We change nothing here. We only look.

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# These notebooks live in notebooks/, so data and models are one level up.
DATA = "../data"
MODELS = "../models"
os.makedirs(MODELS, exist_ok=True)
print("Setup complete.")

## 1. Load the raw file

We use the **Cardiovascular Disease dataset** from Kaggle
(`sulianova/cardiovascular-disease-dataset`) — **70,000 real patient examination records**.

⚠️ **Watch out:** this file is separated by **semicolons (`;`)**, not commas. Forget `sep=";"`
and pandas will squeeze everything into one useless column.

In [ ]:
df = pd.read_csv(f"{DATA}/cardio_train.csv", sep=";")

print(f"Patients : {df.shape[0]:,}")
print(f"Columns  : {df.shape[1]}")
df.head()

## 2. What does each column mean?

| Column | Meaning | Where it comes from |
|---|---|---|
| `id` | Patient number | Just a row label — useless for prediction |
| `age` | Age **in days**, not years | Objective fact |
| `gender` | 1 = woman, 2 = man | Objective fact |
| `height` | Height in cm | Objective fact |
| `weight` | Weight in kg | Objective fact |
| `ap_hi` | Systolic BP (the **upper** number) | Measured by a nurse |
| `ap_lo` | Diastolic BP (the **lower** number) | Measured by a nurse |
| `cholesterol` | 1 = normal, 2 = above, 3 = well above | Lab test |
| `gluc` | Blood sugar: 1 = normal, 2 = above, 3 = well above | Lab test |
| `smoke` | 0 = no, 1 = yes | **Patient said so** |
| `alco` | 0 = no, 1 = yes | **Patient said so** |
| `active` | 0 = no, 1 = yes | **Patient said so** |
| `cardio` | **0 = healthy, 1 = heart disease** | 🎯 What we predict |

The last three are **self-reported** — the patient simply told the doctor. People routinely
under-report smoking and drinking. Remember this: it explains a surprising result in notebook 03.

In [ ]:
print("=== Data types and memory use ===")
df.info()

## 3. Is anything missing?

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values in the whole file: {missing.sum()}")
print("\nZero everywhere — we do not need to fill in any gaps.")

## 4. Is the target balanced?

If 95% of patients were healthy, a lazy model could score 95% by always saying "healthy" and
still be useless. So we check the balance before trusting accuracy as a score.

In [ ]:
counts = df["cardio"].value_counts().sort_index()
print(counts)
print("\nAs a percentage:")
print((counts / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Healthy (0)", "Heart disease (1)"], counts.values,
              color=["#10B981", "#EF4444"], width=0.55)
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_ylabel("Number of patients")
ax.set_title("Almost exactly 50 / 50 — a well balanced dataset")
plt.tight_layout(); plt.show()

print("\nBecause it is balanced we do NOT need SMOTE or class weights,")
print("and 'accuracy' will be a meaningful score.")

## 5. 🚨 The important part — is the data believable?

This is where careless projects go wrong. Let us look at the range of every measurement.

In [ ]:
df[["age", "height", "weight", "ap_hi", "ap_lo"]].describe().T.round(2)

### Look carefully at that table. The data is dirty.

| Problem | What we see | Reality |
|---|---|---|
| `ap_hi` maximum | **16,020** | A human maximum is around 200. Someone typed extra digits. |
| `ap_lo` maximum | **11,000** | Same problem. |
| `ap_lo` minimum | **negative** | Physically impossible. |
| `height` range | 55 cm to 250 cm | Not adult humans. |
| `weight` minimum | 10 kg | Not an adult. |

If we train on this rubbish, the model learns rubbish. **Notebook 02 fixes all of it** — and being
able to *show* your professor these numbers is what separates a serious project from a
copy-pasted one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df["ap_hi"], vert=False)
axes[0].set_title("Systolic BP BEFORE cleaning — look at the scale!")
axes[0].set_xlabel("mmHg")
axes[1].boxplot(df["ap_lo"], vert=False)
axes[1].set_title("Diastolic BP BEFORE cleaning")
axes[1].set_xlabel("mmHg")
plt.tight_layout(); plt.show()

print("The boxes are squashed to the far left because a handful of impossible")
print("values stretch the axis all the way out to 16,000.")

In [ ]:
print("=== How many rows are actually broken? ===")
bad_bp   = ((df.ap_hi < 90) | (df.ap_hi > 200) | (df.ap_lo < 60) | (df.ap_lo > 130)).sum()
bad_order = (df.ap_lo >= df.ap_hi).sum()
bad_body = (~df.height.between(140, 200) | ~df.weight.between(40, 150)).sum()
dupes    = df.duplicated(subset=[c for c in df.columns if c != "id"]).sum()

print(f"Impossible blood pressure values : {bad_bp:,}")
print(f"Lower BP >= upper BP             : {bad_order:,}")
print(f"Impossible height or weight      : {bad_body:,}")
print(f"Duplicate patients               : {dupes:,}")
print("\n(These overlap, so the final removal count will be smaller than the sum.)")

---
## ✅ What we learned

* **70,000 patients, 13 columns**, no missing values.
* The target is **balanced 50/50**, so accuracy is a fair score and no resampling is needed.
* The data contains **medically impossible values** — blood pressure of 16,020 mmHg, negative
  diastolic readings, 55 cm adults.
* Three features (`smoke`, `alco`, `active`) are **self-reported** and therefore less reliable
  than the nurse-measured ones.

### ▶️ Next: `02_Data_Preprocessing.ipynb` — clean all of this up.